[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# Puesta a punto

¡Te damos la bienvenida a LangChain Academy!

## Contexto

En LangChain queremos que construir aplicaciones con LLM sea fácil. Un tipo de aplicación con LLM que puedes construir es un agente. Hay mucho entusiasmo con los agentes porque pueden automatizar una gran variedad de tareas que antes eran imposibles.

En la práctica, sin embargo, es muy difícil construir sistemas que ejecuten esas tareas de forma confiable. Al trabajar con nuestros usuarios para llevar agentes a producción, aprendimos que muchas veces hace falta más control. Puede que necesites que un agente llame siempre primero a una herramienta específica, o que use prompts distintos según su estado.

Para resolver este problema creamos [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), un framework para construir aplicaciones con agentes y multiagente. LangGraph es independiente del paquete LangChain, y su filosofía de diseño es ayudar a los desarrolladores a darles más precisión y control a los flujos de trabajo con agentes, a la altura de la complejidad de los sistemas reales.

## Estructura del curso

El curso está organizado en módulos, y cada módulo se centra en un tema de LangGraph. Verás una carpeta por módulo, con una serie de notebooks. Cada notebook va acompañado de un video que explica los conceptos, pero los notebooks también se entienden solos: traen sus propias explicaciones y puedes leerlos sin los videos. Cada carpeta de módulo incluye además una carpeta `studio` con un conjunto de grafos que se pueden cargar en [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), nuestro IDE para construir aplicaciones con LangGraph.

## Instalación

Antes de empezar, sigue las instrucciones del `README` para crear un entorno e instalar las dependencias.

## Modelos de chat

En este curso usaremos modelos de chat, que reciben una secuencia de mensajes como entrada y devuelven mensajes como salida. LangChain admite muchos modelos mediante [integraciones de terceros](https://docs.langchain.com/oss/python/integrations/chat). Por defecto, el curso usa [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai), porque es popular y rinde bien. Como ya se indicó, asegúrate de tener una `OPENAI_API_KEY`.

Revisemos que tu `OPENAI_API_KEY` esté definida. Si no lo está, se te pedirá que la ingreses.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily

In [2]:
import os, getpass

def _definir_variable(variable: str):
    if not os.environ.get(variable):
        os.environ[variable] = getpass.getpass(f"{variable}: ")

_definir_variable("OPENAI_API_KEY")

[Aquí](https://docs.langchain.com/oss/python/langchain/models) hay una guía útil con todo lo que puedes hacer con los modelos de chat, pero a continuación mostraremos algunos puntos clave. Si ejecutaste `pip install -r requirements.txt` como se indica en el README, ya tienes instalado el paquete `langchain-openai`. Con él podemos crear una instancia del objeto `ChatOpenAI`. Los precios de los distintos modelos están [aquí](https://openai.com/api/pricing/). Los notebooks usan `gpt-4o` por defecto porque ofrece un buen equilibrio entre calidad, precio y velocidad, pero también puedes elegir la serie `gpt-3.5`, más económica, o modelos más recientes.

Hay [algunos parámetros estándar](https://docs.langchain.com/oss/python/langchain/models#parameters) que podemos configurar en los modelos de chat. Dos de los más comunes son:

* `model`: el nombre del modelo
* `temperature`: la temperatura de muestreo

La temperatura (`temperature`) controla qué tan aleatoria o creativa es la salida del modelo. Una temperatura baja (cercana a 0) da salidas más deterministas y enfocadas, lo que sirve para tareas que requieren precisión o respuestas factuales. Una temperatura alta (cercana a 1) sirve para tareas creativas o para generar respuestas variadas.

In [3]:
from langchain_openai import ChatOpenAI
chat_gpt4o = ChatOpenAI(model="gpt-4o", temperature=0)
chat_gpt35 = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

Los modelos de chat de LangChain tienen varios [métodos por defecto](https://reference.langchain.com/python/langchain_core/runnables). Casi siempre usaremos estos:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): devuelve la respuesta por partes (*chunks*)
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): llama a la cadena con una entrada

Y, como dijimos, los modelos de chat reciben [mensajes](https://docs.langchain.com/oss/python/langchain/messages) como entrada. Un mensaje tiene un rol (que indica quién lo dice) y una propiedad `content`. Hablaremos mucho más de esto después; aquí solo mostramos lo básico.

In [4]:
from langchain_core.messages import HumanMessage

# Crear un mensaje
mensaje = HumanMessage(content="Hola mundo", name="Lance")

# Lista de mensajes
mensajes = [mensaje]

# Invocar el modelo con una lista de mensajes
chat_gpt4o.invoke(mensajes)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_eb3c3cb84d', 'id': 'chatcmpl-CSWGCGbWLCLdHUAOh9pkNFWl1MWf4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--529fb9c7-f813-47a7-9662-dea715d4be28-0', usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Obtenemos como respuesta un `AIMessage`. Ten en cuenta que también podemos invocar un modelo de chat con un string. Cuando se pasa un string como entrada, se convierte en un `HumanMessage` y luego se envía al modelo.


In [5]:
chat_gpt4o.invoke("hola mundo")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_cbf1785567', 'id': 'chatcmpl-CSWGCXlVYTEWoHAFm3GXIIgUO1gCb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--43b070a6-7676-4aa1-984c-c0cd43d45c1e-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
chat_gpt35.invoke("hola mundo")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CSWGDY5KePqihRcWDX3IGEZfSoyld', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--f7963c03-f8b2-4eba-bc7f-4d58520fa661-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

La interfaz es la misma en todos los modelos de chat, y normalmente los modelos se inicializan una sola vez al comienzo de cada notebook.

Así, si prefieres otro proveedor, puedes cambiar de modelo sin modificar el resto del código.


## Herramientas de búsqueda

En el README también verás [Tavily](https://tavily.com/), un motor de búsqueda optimizado para LLM y RAG, pensado para dar resultados de búsqueda rápidos, eficientes y persistentes. Como ya se mencionó, registrarse es fácil y tiene un plan gratuito generoso. Algunas lecciones (en el módulo 4) usan Tavily por defecto, pero, por supuesto, puedes usar otras herramientas de búsqueda si modificas el código por tu cuenta.

In [7]:
_definir_variable("TAVILY_API_KEY")

In [11]:
from langchain_tavily import TavilySearch  # actualizado en la versión 1.0

busqueda_tavily = TavilySearch(max_results=3)

datos = busqueda_tavily.invoke({"query": "¿Qué es LangGraph?"})
documentos_busqueda = datos.get("results", datos)

In [12]:
documentos_busqueda

[{'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It?',
  'content': 'LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. By managing the flow of data and the sequence of operations, LangGraph allows developers to focus on the high-level logic of their applications rather than the intricacies of agent coordination. Whether you need a chatbot that can handle various types of user requests or a multi-agent system that performs complex tasks, LangGraph provides the tools to build exactly what you need. LangGraph significantly simplifies the development of complex LLM applications by providing a structured framework for managing state and coordinating agent interactions.',
  'score': 0.9581988,
  'raw_content': None},
 {'url': 'https://www.geeksforgeeks.org/machine-learning